# 07 Model Freeze and Validation

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp017_freeze_and_archive.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_freeze_and_archive.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp017_frozen_officialval_five_scale_audit.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_frozen_officialval_five_scale_audit.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp017_locked_team_validation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_locked_team_validation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Normal/scripts/exp017_freeze_and_archive.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a7634761146b4a80486bdaf5073d44d45ed897dc'}, '05_model_optimize/MLP/Normal/scripts/exp017_frozen_officialval_five_scale_audit.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '565b7630e4c88b5d5cea6397c8934308e341fe9f'}, '05_model_optimize/MLP/Normal/scripts/exp017_locked_team_validation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2f838cebe19bec3321c890eb90f2b878c7443e46'}}

## exp017_freeze_and_archive

원본: [05_model_optimize/MLP/Normal/scripts/exp017_freeze_and_archive.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_freeze_and_archive.py)  
Git blob: `a7634761146b4a80486bdaf5073d44d45ed897dc`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Archive EXP017 OfficialVal-only results and freeze winning 3-seed joblibs.

Do not train or load team handwriting. This is a copy-and-SHA-verification step.
No main/final model mutation. The workflow controls git commit target branch.



In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_freeze_and_archive.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Archive EXP017 OfficialVal-only results and freeze winning 3-seed joblibs.

Do not train or load team handwriting. This is a copy-and-SHA-verification step.
No main/final model mutation. The workflow controls git commit target branch.
"""
from __future__ import annotations
import argparse,json,hashlib,shutil
from pathlib import Path
from exp017_frozen_officialval_five_scale_audit import path_for

def copy_tree(src,dst):
    src=Path(src);dst=Path(dst)
    assert src.is_dir() and not dst.exists(),(src,dst)
    dst.mkdir(parents=True)
    for f in src.iterdir():
        assert f.is_file(),f
        shutil.copy2(f,dst/f.name)

def run(coarse_review,final_review,audit_review,coarse_models,fine_models,old_models,archive_root,model_root):
    archive_root=Path(archive_root);model_root=Path(model_root)
    assert not archive_root.exists() and not model_root.exists()
    archive_root.mkdir(parents=True);model_root.mkdir(parents=True)
    copy_tree(coarse_review,archive_root/"coarse_26config_3seed")
    copy_tree(final_review,archive_root/"joint_final_46config_3seed")
    copy_tree(audit_review,archive_root/"frozen_officialval_five_scales")
    pick=json.loads((Path(audit_review)/"SELECTION.json").read_text(encoding="utf-8"))
    selected=pick.get("new_candidate")
    if selected is None:
        (model_root/"README.md").write_text("# EXP유017 동결 후보 없음\n\n안전 기준을 통과한 후보가 없어 새 모델을 선택하지 않았다. 팀 손글씨는 평가하지 않았다.\n",encoding="utf-8")
        print("EXP017_NO_NEW_FROZEN_CANDIDATE",flush=True);return
    joint=json.loads((Path(final_review)/"metadata.json").read_text(encoding="utf-8"))
    chosen=next((v for v in (joint["selected_frozen_candidate"],joint["runner_up"])
                 if v and v["variant"]==selected),None)
    assert chosen is not None
    expected={(r["label"],int(r["seed"])):r for r in pick["checkpoints"]}
    archive=[]
    for seed in (42,43,44):
        old=path_for(chosen,seed,coarse_models,fine_models,old_models)
        assert old.is_file()
        digest=hashlib.sha256(old.read_bytes()).hexdigest()
        assert expected[pick["new_candidate_label"],seed]["sha256"]==digest,(seed,digest)
        dst=model_root/f"seed{seed}.joblib"
        shutil.copy2(old,dst)
        assert hashlib.sha256(dst.read_bytes()).hexdigest()==digest
        archive.append({"seed":seed,"file":dst.name,"sha256":digest,
               "bytes":dst.stat().st_size,"source_variant":selected,
               "source_study":chosen["source"],"source_action_artifact":str(old),
               "team_handwriting_used":False})
    data={"experiment":"EXP017","stage":"frozen_hyperparameters_before_team_validation",
        "chosen_variant":selected,"source":chosen["source"],
        "hidden":chosen["hidden"],"alpha":chosen["alpha"],
        "learning_rate_init":chosen["learning_rate_init"],
        "early_stopping":chosen["early_stopping"],
        "preprocessing":"80pct_bbox_size_14pct_stroke_training__com_inference",
        "official_source_sha256":joint["official_val_source_sha256"],
        "checkpoints":archive,
        "official_val_only_hyperparam_selection":True,
        "team_handwriting_evaluated":False,
        "original_test_used":False,"professor_test_used":False,
        "final_deployment_performed":False}
    (model_root/"MANIFEST.json").write_text(json.dumps(data,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    (model_root/"README.md").write_text(
        "# EXP유017 — 80-14 고정 MLP 하이퍼파라미터 최적화 동결 후보\n\n"+
        f"- 선택된 모델: **{selected}**\n"+
        f"- 은닉층: **{chosen['hidden']}**, alpha={chosen['alpha']}, learning_rate_init={chosen['learning_rate_init']}, early_stopping={chosen['early_stopping']}\n"+
        "- 각 Seed (42·43·44)의 학습된 joblib 3개를 원본 Actions SHA-256과 동일하게 보존\n"+
        "- 기존 70-0 / 80-14 동결 체크포인트는 그대로 유지, main 및 final 체크포인트 변경 없음\n"+
        "- **팀 손글씨 Validation 미수행**. 이 모델은 독립 손글씨 평가를 위한 가중치 동결 후보이며 최종 배포 모델 아님\n"+
        "- [실측 46조합 및 5배율 결과](../크기편향_원인분석/EXP유017_하이퍼파라미터_실측/)\n",
        encoding="utf-8")
    print("EXP017_THREE_SEED_FREEZE_COMPLETE",selected,flush=True)
if __name__=="__main__":
    a=argparse.ArgumentParser()
    for nm in ("coarse-review","final-review","audit-review","coarse-models","fine-models","old-models","archive-root","model-root"):
        a.add_argument("--"+nm,type=Path,required=True)
    x=a.parse_args()
    run(x.coarse_review,x.final_review,x.audit_review,x.coarse_models,x.fine_models,
        x.old_models,x.archive_root,x.model_root)


## exp017_frozen_officialval_five_scale_audit

원본: [05_model_optimize/MLP/Normal/scripts/exp017_frozen_officialval_five_scale_audit.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_frozen_officialval_five_scale_audit.py)  
Git blob: `565b7630e4c88b5d5cea6397c8934308e341fe9f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

OfficialVal 5-scale final audit of frozen EXP017 winners; no training/handwriting.

Include exact prior EXP016 80-14 checkpoints for paired scale benchmark.

CLI 인자 (정적 추출): `--metadata`, `--coarse-models`, `--fine-models`, `--old-models`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_frozen_officialval_five_scale_audit.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""OfficialVal 5-scale final audit of frozen EXP017 winners; no training/handwriting.

Include exact prior EXP016 80-14 checkpoints for paired scale benchmark.
"""
from __future__ import annotations
import argparse,csv,json,hashlib
from pathlib import Path
import numpy as np,joblib
from exp006_size_com_comparison import ROOT,LABELS,process,load_official,image_geometry
from exp011_012_causal_probes import fixed_scale
from exp017_hparam_coarse import CONF,config_name
SCALES=(.8,.9,1.,1.1,1.2)
SEEDS=(42,43,44)
FOCUS={"B","8","D"}
def path_for(meta,seed,coarse,fine,old):
    if meta is None:
        return Path(old)/f"exp016-s{seed}-r80"/"size80_stroke14.joblib"
    name=meta["variant"]
    if meta["source"]=="coarse":
        found=[i for i,(h,a,lr) in enumerate(CONF) if config_name(h,a,lr)==name]
        assert len(found)==1
        return Path(coarse)/f"exp017-coarse-s{seed}-p{found[0]%3}"/(name+".joblib")
    assert meta["source"]=="fine"
    slot=int(name.split("_",1)[0].replace("slot",""))-1
    assert slot in (0,1,2,3)
    return Path(fine)/f"exp017-fine-s{seed}-slot{slot}"/(name+".joblib")
def write(path,rows):
    assert rows
    with Path(path).open("w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def run(metadata_path,coarse,fine,old,out):
    doc=json.loads(Path(metadata_path).read_text(encoding="utf-8"))
    assert doc["experiment"]=="EXP017" and doc["stage"]=="joint_final"
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    if doc["selected_frozen_candidate"] is None:
        (out/"SELECTION.json").write_text(json.dumps({"selected":None,"reason":"No candidate passed original predeclared gates"},indent=2))
        print("EXP017_NO_GATE_PASS_UNCHANGED",flush=True);return
    official=load_official(ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz")
    y=np.asarray(official["y_val"],dtype=int);X=official["X_val"]
    assert X.shape==(19200,784) and set(np.unique(y))==set(range(16))
    target=int(np.median(image_geometry(official["X_train"])["long_side"]))
    assert target==20
    scaled={s:process(X if s==1. else fixed_scale(X,s),"com",target) for s in SCALES}
    contenders=[("rank1",doc["selected_frozen_candidate"])]
    if doc["runner_up"] is not None:contenders.append(("rank2",doc["runner_up"]))
    contenders.append(("incumbent_80_14",None))
    metrics=[];checkpoints=[]
    for label,meta in contenders:
        for seed in SEEDS:
            path=path_for(meta,seed,coarse,fine,old)
            assert path.exists(),path
            digest=hashlib.sha256(path.read_bytes()).hexdigest()
            model=joblib.load(path)
            probabilities={};pred={}
            for s in SCALES:
                probs=np.asarray(model.predict_proba(scaled[s]),dtype=np.float32)
                assert probs.shape==(19200,16)
                probabilities[s]=probs[:,10:].sum(axis=1)
                pred[s]=np.argmax(probs,axis=1)
            base=probabilities[1.]>=.5
            masks={"all":np.ones(len(y),dtype=bool),"digits":y<10,"letters":y>=10}
            masks.update({"class_"+cl:y==i for i,cl in enumerate(LABELS)})
            for scale in SCALES:
                for group,m in masks.items():
                    p=probabilities[scale][m];q=probabilities[1.][m]
                    metrics.append({"candidate":label,
                      "variant":meta["variant"] if meta else "size80_stroke14",
                      "seed":seed,"scale":scale,"group":group,"n":int(np.sum(m)),
                      "accuracy":float(np.mean(pred[scale][m]==y[m])),
                      "family_flip":float(np.mean((p>=.5)!=base[m])),
                      "mean_abs_prob_delta":float(np.mean(np.abs(p-q)))})
            checkpoints.append({"label":label,"seed":seed,"variant":meta["variant"] if meta else "size80_stroke14",
                "source_path":str(path),"sha256":digest,"bytes":path.stat().st_size,
                "fitting_handmade_used":False})
            print("EXP017_FROZEN_AUDIT",label,seed,"val_accuracy",
                 next(r["accuracy"] for r in metrics if r["candidate"]==label and r["seed"]==seed and r["scale"]==1. and r["group"]=="all"),flush=True)
            del model
    write(out/"frozen_models_all_seed_five_scale_metrics.csv",metrics)
    def measure(label,group,sc,field):
        x=[r[field] for r in metrics if r["candidate"]==label and r["group"]==group and abs(r["scale"]-sc)<1e-9]
        assert len(x)==3,(label,group,sc,field,len(x))
        return sum(x)/len(x)
    summary=[]
    for label,meta in contenders:
        focal=max((measure(label,"class_"+cl,sc,"family_flip"),cl,sc)
                for cl in FOCUS for sc in (.8,.9,1.1,1.2))
        allmax=max((measure(label,"class_"+cl,sc,"family_flip"),cl,sc)
                for cl in LABELS for sc in (.8,.9,1.1,1.2))
        digit12=measure(label,"digits",1.2,"family_flip")
        letter08=measure(label,"letters",.8,"family_flip")
        val=measure(label,"all",1.,"accuracy")
        passed=(focal[0]<=.20+1e-10 and allmax[0]<=.25+1e-10 and digit12<=.10+1e-10
                and letter08<=.10+1e-10 and val>=.9431076388888888-1e-10
                and measure(label,"digits",1.2,"mean_abs_prob_delta")<=.12+1e-10
                and measure(label,"letters",.8,"mean_abs_prob_delta")<=.12+1e-10)
        summary.append({"candidate":label,"variant":meta["variant"] if meta else "size80_stroke14",
            "val_accuracy":val,"val_accuracy_08":measure(label,"all",.8,"accuracy"),
            "val_accuracy_12":measure(label,"all",1.2,"accuracy"),
            "digit_12_flip":digit12,"letter_08_flip":letter08,
            "B8D_worst_flip_full5scale":focal[0],"B8D_worst_class":focal[1],
            "B8D_worst_scale":focal[2],"all16_worst_flip_full5scale":allmax[0],
            "all16_worst_class":allmax[1],"all16_worst_scale":allmax[2],
            "passed_five_scale_audit":passed})
    write(out/"frozen_3seed_model_comparison.csv",summary)
    acceptable=[x for x in summary if x["candidate"] in ("rank1","rank2") and x["passed_five_scale_audit"]]
    acceptable.sort(key=lambda x:(-x["val_accuracy"],x["B8D_worst_flip_full5scale"]))
    winner=acceptable[0] if acceptable else None
    select={"experiment":"EXP017","stage":"frozen_officialval_five_scale_audit",
      "new_candidate":winner["variant"] if winner else None,
      "new_candidate_label":winner["candidate"] if winner else None,
      "source":next((meta["source"] for label,meta in contenders if winner and label==winner["candidate"]),None),
      "source_sha256":official["sha256"],"team_handwriting_evaluated":False,
      "original_test_used":False,"professor_test_used":False,
      "checkpoints":checkpoints,"comparison":summary,"predeclared_joint_gate_already_passed":True,
      "five_scale_supplementary_gate_pass":winner is not None}
    (out/"SELECTION.json").write_text(json.dumps(select,ensure_ascii=False,indent=2),encoding="utf-8")
    text=["# EXP유017 동결 모델 OfficialVal 5배율 교차검증","","| 후보 | 1.0 Val | 0.8 Val | 1.2 Val | B·8·D 최악 | 16클래스 최악 | 5배율 Gate |",
         "|---|---:|---:|---:|---:|---:|---|"]
    for r in summary:
        text.append(f"| {r['variant']} | "+ " | ".join(f"{r[k]*100:.2f}%" for k in 
            ("val_accuracy","val_accuracy_08","val_accuracy_12","B8D_worst_flip_full5scale","all16_worst_flip_full5scale"))+
            " | "+("통과" if r["passed_five_scale_audit"] else "탈락")+" |")
    text+=["",f"동결 후보: **{winner['variant'] if winner else '안전 기준 통과 후보 없음'}**.",
           "팀원이 모은 손글씨를 아직 사용하지 않았으며, 교수님 Test/Original Test도 사용하지 않음.",
           "팀 손글씨 실험 전 이 결과와 가중치만 저장하고 선택 기준을 고정해야 한다."]
    (out/"SUMMARY.md").write_text("\n".join(text)+"\n",encoding="utf-8")
    print("EXP017_FIVE_SCALE_COMPLETE",winner["variant"] if winner else "NONE",flush=True)
if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--metadata",type=Path,required=True)
    p.add_argument("--coarse-models",type=Path,required=True)
    p.add_argument("--fine-models",type=Path,required=True)
    p.add_argument("--old-models",type=Path,required=True)
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    run(a.metadata,a.coarse_models,a.fine_models,a.old_models,a.out)


## exp017_locked_team_validation

원본: [05_model_optimize/MLP/Normal/scripts/exp017_locked_team_validation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_locked_team_validation.py)  
Git blob: `2f838cebe19bec3321c890eb90f2b878c7443e46`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Post-freeze team handwriting validation ONLY. Do not run during EXP017 tuning.

Input .npz: X [N,784] (0..255), y [N] with labels 0..15,
optional writer_id [N]. Validate unseen handwritten sheets without refitting.
No Original Test or professor Test. Includes paired 0.8-1.2x interventions.

CLI 인자 (정적 추출): `--npz`, `--frozen-models`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_locked_team_validation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Post-freeze team handwriting validation ONLY. Do not run during EXP017 tuning.

Input .npz: X [N,784] (0..255), y [N] with labels 0..15,
optional writer_id [N]. Validate unseen handwritten sheets without refitting.
No Original Test or professor Test. Includes paired 0.8-1.2x interventions.
"""
from __future__ import annotations
import argparse,csv,json,hashlib
from pathlib import Path
import numpy as np,joblib
from exp006_size_com_comparison import LABELS,process
from exp011_012_causal_probes import fixed_scale
SCALES=(.8,.9,1.,1.1,1.2)
def csv_write(path,rs):
    assert rs
    with Path(path).open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=list(rs[0]));w.writeheader();w.writerows(rs)
def run(data,model_folder,out):
    out=Path(out);data=Path(data);model_folder=Path(model_folder)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    with np.load(data,allow_pickle=False) as npz:
        assert "X" in npz and "y" in npz
        X=np.array(npz["X"],dtype=np.float32)
        y=np.array(npz["y"],dtype=int)
        writers=(np.array(npz["writer_id"]).astype(str)
                 if "writer_id" in npz else np.array(["unknown"]*len(y)))
    assert X.ndim==2 and X.shape[1]==784
    assert np.isfinite(X).all() and np.min(X)>=0 and np.max(X)<=255
    assert y.shape==(len(X),) and writers.shape==(len(X),)
    assert np.isin(y,np.arange(16)).all()
    assert len(np.unique(y))==16,"every class 0..15 required"
    prepared={s:process(X if s==1. else fixed_scale(X,s),"com",20)
              for s in SCALES}
    ckpts=sorted(model_folder.glob("seed*.joblib"))
    assert len(ckpts)==3,ckpts
    records=[];pair=[];loaded=[]
    for path in ckpts:
        import re
        z=re.fullmatch(r"seed(42|43|44)\.joblib",path.name)
        assert z is not None,path.name
        seed=int(z.group(1))
        mdl=joblib.load(path)
        pf={};yp={}
        for s in SCALES:
            prob=np.asarray(mdl.predict_proba(prepared[s]),dtype=np.float32)
            assert prob.shape==(len(y),16)
            pf[s]=np.sum(prob[:,10:],axis=1)
            yp[s]=np.argmax(prob,axis=1)
        baseline=(pf[1.]>=.5)
        masks={"all":np.ones(len(y),bool),"digits":y<10,"letters":y>=10}
        for i,c in enumerate(LABELS):masks["class_"+c]=y==i
        for w in np.unique(writers):
            masks["writer_"+str(w)]=writers==w
        for scale in SCALES:
            for group,mask in masks.items():
                if not mask.any():continue
                records.append({"seed":seed,"scale":scale,"group":group,"n":int(mask.sum()),
                    "accuracy":float(np.mean(yp[scale][mask]==y[mask])),
                    "family_flip":float(np.mean((pf[scale][mask]>=.5)!=baseline[mask])),
                    "mean_abs_delta_p_letter":float(np.mean(np.abs(pf[scale][mask]-pf[1.][mask])))})
        for actual,pred in (("B","8"),("8","B"),("D","0"),("0","D"),("F","5"),("5","F")):
            m=y==LABELS.index(actual)
            pair.append({"seed":seed,"actual":actual,"predicted":pred,
                "count":int(np.sum(yp[1.][m]==LABELS.index(pred)))})
        loaded.append({"seed":seed,"filename":path.name,"sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
    csv_write(out/"team_validation_metrics_all_seed.csv",records)
    csv_write(out/"team_validation_confusion_pairs.csv",pair)
    summ=[]
    for group in ("all","digits","letters"):
        for scale in SCALES:
            values=[r["accuracy"] for r in records if r["group"]==group and r["scale"]==scale]
            flips=[r["family_flip"] for r in records if r["group"]==group and r["scale"]==scale]
            assert len(values)==3
            summ.append({"group":group,"scale":scale,
                "accuracy_seed_mean":float(np.mean(values)),
                "family_flip_seed_mean":float(np.mean(flips)),
                "accuracy_seed_sd":float(np.std(values,ddof=1))})
    csv_write(out/"team_validation_summary.csv",summ)
    manifest={"experiment":"EXP017","stage":"locked_team_validation",
      "dataset_sha256":hashlib.sha256(data.read_bytes()).hexdigest(),
      "n_images":len(y),"n_unique_writers":len(np.unique(writers)),
      "per_class_counts":{cl:int(np.sum(y==i)) for i,cl in enumerate(LABELS)},
      "frozen_checkpoints":loaded,"inference_only":True,
      "trained_or_tuned_on_current_data":False,
      "original_test_used":False,"professor_test_used":False,
      "disclaimer":"If images overlap EXP014-016 Handmade, this is reused development data, not independent blind holdout."}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("EXP017_LOCKED_TEAM_VALIDATION_DONE",len(y),len(np.unique(writers)),flush=True)
if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--npz",required=True,type=Path,help="New team handwriting X[N,784], y[N], optional writer_id[N]")
    ap.add_argument("--frozen-models",required=True,type=Path,help="seed42.joblib, seed43.joblib, seed44.joblib")
    ap.add_argument("--out",required=True,type=Path)
    a=ap.parse_args()
    run(a.npz,a.frozen_models,a.out)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")